In [0]:
from pyspark.sql.functions import col, when, window, sum as _sum, avg, round
from pyspark.sql.window import Window

## Read data from Silver

In [0]:
df_silver_trades = spark.readStream.table("workspace.silver.trades")
df_silver_kline = spark.read.table("workspace.silver.kline")

## Technical Indicators & Algorithmic Trading Signals

In [0]:
window_spec = Window.partitionBy("symbol").orderBy("kline_timestamp")

# 15 nến
window_15 = window_spec.rowsBetween(-14, 0) 
# 50 nến
window_50 = window_spec.rowsBetween(-49, 0)

# Tính toán chỉ báo và tạo tín hiệu
df_gold = df_silver_kline \
    .withColumn("volatility_pct", round(((col("high") - col("low")) / col("open")) * 100, 3)) \
    .withColumn("SMA_15", round(avg("close").over(window_15), 2)) \
    .withColumn("SMA_50", round(avg("close").over(window_50), 2)) \
    .withColumn("trading_signal",
        when(col("SMA_15") > col("SMA_50"), "BULLISH (BUY)")
        .when(col("SMA_15") < col("SMA_50"), "BEARISH (SELL)")
        .otherwise("HOLD (NEUTRAL)")
    ) \
    .dropna() # Loại bỏ những dòng đầu tiên chưa đủ 50 nến để tính SMA

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.kline_signals")
)

print("Đã tính toán xong các chỉ báo tài chính và cập nhật bảng: workspace.gold.kline_signals")

## WHALE ALERTS

### Calculating Trade value

In [0]:
df_trades_enriched = df_silver_trades.withColumn("trade_value", col("price") * col("qty"))

In [0]:
df_whale_alerts = df_trades_enriched \
    .filter(col("trade_value") >= 10000) \
    .withColumn("action", when(col("is_buyer_maker") == True, "SELL").otherwise("BUY")) \
    .select("trade_timestamp", "symbol", "action", "price", "qty", "trade_value")

whale_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/04_gold/whale_alerts"

query_whales = (df_whale_alerts.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", whale_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.gold.whale_alerts")
)

query_whales.awaitTermination()
print("Đã cập nhật bảng: workspace.gold.whale_alerts")

## CALCULATING BUY/SELL PRESSURE (CVD)

In [0]:
df_volume_split = df_trades_enriched \
    .withColumn("buy_vol", when(col("is_buyer_maker") == False, col("trade_value")).otherwise(0)) \
    .withColumn("sell_vol", when(col("is_buyer_maker") == True, col("trade_value")).otherwise(0))

df_cvd_minute = df_volume_split \
    .withWatermark("trade_timestamp", "2 minutes") \
    .groupBy(
        window(col("trade_timestamp"), "1 minute").alias("time_window"), 
        col("symbol")
    ) \
    .agg(
        _sum("buy_vol").alias("total_buy_vol"),
        _sum("sell_vol").alias("total_sell_vol"),
        _sum("trade_value").alias("total_volume")
    ) \
    .withColumn("cvd", col("total_buy_vol") - col("total_sell_vol")) \
    .select("time_window.start", "symbol", "total_buy_vol", "total_sell_vol", "total_volume", "cvd")
    

In [0]:
cvd_checkpoint = "/Volumes/workspace/bronze/binance_raw/_checkpoints/04_gold/cvd_minute"

query_cvd = (df_cvd_minute.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", cvd_checkpoint)
    .trigger(availableNow=True)
    .toTable("workspace.gold.minute_order_flow")
)

query_cvd.awaitTermination()
print("Đã cập nhật bảng: workspace.gold.minute_order_flow")

In [0]:
%sql
-- SELECT * FROM workspace.gold.whale_alerts

In [0]:
%sql
-- SELECT * FROM workspace.gold.minute_order_flow